![Pythonista](img/pythonista.png)

# *YAML* y sintaxis de *workflows*.

Este cuaderno desarrolla *YAML*, *triggers*, *jobs*, *steps*, expresiones y variables con el formato editorial de Pythonista.

## Objetivos.

- Entender la estructura mínima de un *workflow*.
- Identificar los *triggers* principales.
- Reconocer el papel de expresiones y contextos.
- Encadenar *jobs* con `needs:` y condicionarlos con `if:`.
- Persistir resultados entre *jobs* con artefactos.
- Controlar *timeouts* y fallos esperados.

## Estructura mínima de un *workflow*.

Un *workflow* se define dentro de `.github/workflows/` y normalmente contiene bloques como `name`, `on`, `jobs` y `steps`. También usaremos expresiones `${{ ... }}` y contextos como `github`, `env`, `vars` y `secrets`.

```yaml
name: ci
on:
  push:
    branches: ["main"]
  pull_request:
jobs:
  checks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - run: echo ${{ github.ref }}
```

## Qué es un *workflow* y cómo se lee.

Un *workflow* es la unidad de automatización de *GitHub Actions*. Se dispara cuando ocurre uno o varios eventos definidos en `on` y organiza el trabajo que debe ejecutarse en respuesta a esos eventos.

Conviene leer un *workflow* de arriba hacia abajo en este orden:

- `name`: etiqueta legible de la automatización.
- `on`: condiciones que la activan.
- `jobs`: trabajos independientes o encadenados.
- `runs-on`: tipo de *runner* requerido por cada *job*.
- `steps`: secuencia de acciones y comandos dentro del *job*.

La idea importante es que un *workflow* no es un *script* lineal grande. Es una declaración de condiciones, dependencias y pasos de ejecución. Eso permite reusar acciones, paralelizar trabajos y aplicar reglas distintas según rama, evento o entorno.

## *Triggers*, contextos y expresiones.

Los *triggers* responden a eventos del repositorio, por ejemplo `push`, `pull_request`, `workflow_dispatch` o `schedule`. A partir de ese evento, *GitHub* expone datos en distintos contextos como `github`, `runner`, `env`, `vars`, `secrets` y `matrix`.

Las expresiones `${{ ... }}` permiten usar esos datos para tomar decisiones dentro del *workflow*. Por ejemplo, habilitar un despliegue solo si la rama es `main`, cambiar variables por entorno o construir una matriz de versiones.

Esa combinación de evento, contexto y expresión es lo que vuelve a *GitHub Actions* un DSL operativo para *CI/CD* y no solo un archivo *YAML* descriptivo.

## Parametrización: valores fuera del código.

Un *workflow* bien diseñado no contiene valores concretos en el código. Nombres de proyectos, regiones, versiones de herramientas o credenciales deben vivir fuera del *YAML* y entrar como parámetros. Este principio tiene consecuencias directas en seguridad, reutilización y mantenibilidad.

### Por qué parametrizar.

- **Seguridad**: un secreto en el código es un secreto expuesto. Una credencial hardcodeada en un *workflow* permanece en el historial de *Git* aunque se elimine más tarde.
- **Paridad de entornos**: el mismo *workflow* funciona en `test` y en `prod` cambiando el valor de una variable, no el código.
- **Trazabilidad**: los valores externos están en un único lugar; cambiarlos no requiere editar *YAML*.
- **Reutilización**: un *workflow* sin valores hardcodeados puede usarse como plantilla en otros proyectos.

### Los tres mecanismos de *GitHub Actions*.

| Mecanismo | Acceso | Uso típico | Visible en logs |
|---|---|---|---|
| `vars.*` | `${{ vars.NOMBRE }}` | Valores no sensibles por entorno (región, proyecto, URL) | Sí |
| `secrets.*` | `${{ secrets.NOMBRE }}` | Credenciales, tokens, contraseñas | No (enmascarado) |
| `inputs.*` | `${{ inputs.NOMBRE }}` | Valores que elige quien dispara el *workflow* manualmente | Sí |

```yaml
jobs:
  deploy:
    runs-on: ubuntu-latest
    environment: prod
    steps:
      # Mal: valor hardcodeado
      - run: gcloud run deploy mi-servicio --region us-central1

      # Bien: valor parametrizado por entorno
      - run: gcloud run deploy ${{ vars.SERVICE_NAME }} --region ${{ vars.GCP_REGION }}

      # Credencial nunca en el YAML
      - run: echo ${{ secrets.API_TOKEN }} | docker login
```

`vars.*` y `secrets.*` se configuran en `Settings → Environments` del repositorio y pueden tener valores distintos por entorno: `test` apunta a una región de desarrollo, `prod` apunta a la región de producción. El *workflow* no cambia; cambian los valores que recibe.

### Inputs para *workflow_dispatch*.

Cuando un *workflow* se dispara manualmente, `inputs` permite que quien lo lanza elija valores en el momento:

```yaml
on:
  workflow_dispatch:
    inputs:
      entorno:
        description: "Entorno destino"
        required: true
        type: choice
        options: [test, prod]
      version:
        description: "Versión a desplegar"
        required: true
        type: string
```

Esto convierte el *workflow* en una operación controlada: el operador elige el entorno y la versión sin editar el código. El mismo *YAML* soporta múltiples escenarios de ejecución.

## Ejemplo concreto: *workflow* para *pull request* y rama principal.

En un repositorio protegido, un flujo común es este: una rama corta dispara *checks* con `pull_request`; después del *merge* a `main`, otro evento `push` habilita pasos de promoción o despliegue controlado.

```yaml
on:
  pull_request:
    branches: ["main"]
  push:
    branches: ["main"]
```

Este patrón conecta directamente el *workflow* técnico con la política del repositorio: el *pull request* valida; la rama principal promueve.

## Dependencias entre *jobs*.

Por defecto, todos los *jobs* de un *workflow* se ejecutan en paralelo. Para que un *job* espere a que otro termine antes de arrancar, se usa `needs:`.

```yaml
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - run: ruff check .

  tests:
    runs-on: ubuntu-latest
    needs: lint          # solo corre si lint termina con éxito
    steps:
      - uses: actions/checkout@v6
      - run: pytest

  deploy:
    runs-on: ubuntu-latest
    needs: [lint, tests] # espera a ambos
    steps:
      - run: echo "desplegando"
```

Esto define un grafo de dependencias: `lint` y `tests` pueden correr en paralelo; `deploy` espera a que ambos terminen con éxito. Si cualquier *job* del que se depende falla, los *jobs* descendientes se cancelan automáticamente.

`needs:` acepta un solo nombre (`needs: lint`) o una lista (`needs: [lint, tests]`). Un *job* sin `needs:` siempre corre en paralelo con los demás.

Referencias de herramientas nuevas en este bloque: [*ruff*](https://docs.astral.sh/ruff/) y [*pytest*](https://docs.pytest.org/).

## Condicionales: `if:` en *jobs* y *steps*.

`if:` permite ejecutar un *job* o un *step* solo cuando se cumple una condición. Se evalúa como expresión `${{ ... }}`, aunque los corchetes son opcionales cuando no hay ambigüedad.

```yaml
jobs:
  deploy:
    runs-on: ubuntu-latest
    needs: tests
    if: github.ref == 'refs/heads/main'   # solo en main
    steps:
      - uses: actions/checkout@v6

      - name: notificar en fallo
        if: failure()                         # corre si un step anterior falló
        run: echo "algo salió mal"

      - name: limpiar siempre
        if: always()                          # corre sin importar el resultado
        run: rm -rf /tmp/build
```

Las funciones de estado más útiles son:

| Función | Cuándo corre |
|---|---|
| `success()` | todos los *steps* anteriores pasaron (comportamiento por defecto) |
| `failure()` | al menos un *step* anterior falló |
| `always()` | sin importar el resultado |
| `cancelled()` | el *workflow* fue cancelado |

Combinado con `needs:`, `if:` permite construir flujos completos: desplegar solo si los tests pasan y la rama es `main`, notificar solo si algo falla, limpiar recursos siempre.

## Artefactos: persistir resultados entre *jobs*.

Cada *job* corre en un *runner* efímero que desaparece al terminar. Si un *job* genera un archivo que otro necesita —un binario compilado, un reporte de cobertura, una imagen exportada— debe persistirlo como artefacto antes de que el *runner* se elimine.

```yaml
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - run: python -m build        # genera dist/
      - uses: actions/upload-artifact@v4
        with:
          name: paquete
          path: dist/

  publish:
    runs-on: ubuntu-latest
    needs: build
    steps:
      - uses: actions/download-artifact@v4
        with:
          name: paquete
          path: dist/
      - run: twine upload dist/*
```

`upload-artifact` guarda los archivos en el almacenamiento de *GitHub*. `download-artifact` los recupera en cualquier *job* posterior. Los artefactos se eliminan automáticamente a los 90 días por defecto (configurable con `retention-days:`).

Este patrón tiene una consecuencia de seguridad directa: garantiza que lo que se probó y lo que se despliega es exactamente el mismo binario, no una reconstrucción posterior.

Referencias de herramientas nuevas en este bloque: [*actions/upload-artifact*](https://github.com/actions/upload-artifact), [*actions/download-artifact*](https://github.com/actions/download-artifact) y [*twine*](https://twine.readthedocs.io/).

## Matriz de versiones.

La estrategia `matrix` ejecuta el mismo *job* con distintos valores sin duplicar código. Es la forma idiomática de validar contra múltiples versiones de *Python*, sistemas operativos o configuraciones:

```yaml
jobs:
  tests:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12", "3.13"]
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with:
          python-version: ${{ matrix.python-version }}
      - run: pytest
```

*GitHub Actions* crea un *job* independiente por cada valor y los ejecuta en paralelo. Con dos dimensiones —versión de *Python* y sistema operativo— el número de *jobs* es el producto cartesiano de ambas listas.

La matriz se cubre en detalle en capítulos posteriores junto con su integración con entornos protegidos y artefactos.

## Control de ejecución: *timeouts* y fallos esperados.

`timeout-minutes:` limita cuánto puede correr un *job* o un *step*. Sin este límite, un proceso colgado consume tiempo de *runner* hasta agotar la cuota del repositorio:

```yaml
jobs:
  tests:
    runs-on: ubuntu-latest
    timeout-minutes: 15        # el job falla si no termina en 15 min
    steps:
      - uses: actions/checkout@v6
      - run: pytest
        timeout-minutes: 10    # también aplicable a steps individuales
```

`continue-on-error:` permite que un *job* o *step* falle sin cancelar el resto del *workflow*. Útil para *checks* opcionales que deben reportar pero no bloquear:

```yaml
      - name: escaneo de seguridad (no bloqueante)
        continue-on-error: true
        run: trivy fs .
```

Usados junto con `if: failure()`, estos controles permiten construir *pipelines* que reportan información aunque algo falle, en lugar de cancelarlo todo ante el primer error.

Referencia de herramienta nueva en este bloque: [*Trivy*](https://trivy.dev/).

## *Pipeline* completo: integrando los conceptos aprendidos.

El siguiente *workflow* combina *triggers*, dependencias entre *jobs*, condicionales, artefactos y control de ejecución en un *pipeline* realista de *CI/CD*:

```yaml
name: ci-cd

on:
  pull_request:
  push:
    branches: ["main"]

jobs:
  lint:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v6
      - run: pip install ruff
      - run: ruff check .

  tests:
    runs-on: ubuntu-latest
    needs: lint
    timeout-minutes: 15
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -r requirements.txt
      - run: pytest
      - uses: actions/upload-artifact@v4
        if: always()                          # guardar reporte aunque los tests fallen
        with:
          name: reporte-${{ matrix.python-version }}
          path: .coverage
          retention-days: 7

  deploy:
    runs-on: ubuntu-latest
    needs: [lint, tests]
    if: github.ref == 'refs/heads/main'       # solo en main, no en PRs
    environment: staging
    steps:
      - uses: actions/checkout@v6
      - run: echo "desplegando a staging"

      - name: notificar si falló
        if: failure()
        run: echo "el despliegue falló"
```

Este *pipeline* expresa las siguientes decisiones:

- `lint` y `tests` forman una cadena secuencial; `deploy` espera a ambos.
- Los tests corren en paralelo para *Python* 3.11 y 3.12 gracias a la *matrix*.
- El reporte de cobertura se persiste como artefacto aunque los tests fallen (`if: always()`).
- El despliegue solo ocurre en `main`, nunca en un *pull request*.
- El *job* `deploy` está asociado a un entorno (`environment: staging`) —en este caso, *staging* es un ejemplo de nomenclatura común; cada organización define sus propios entornos y nombres según su contexto.
- *GitHub* aplicará las reglas de aprobación definidas para el entorno seleccionado antes de permitir que el *job* corra.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>